# 65 — Compare the two render paths

Decision D8 keeps two routes from the export to RDF and gives them different jobs:
the **direct renderer** (`50_render_bc.ipynb`) produces the deliverable, and
**`linkml-convert`** over a patched schema is kept as an independent check.

This notebook runs both on a sample of real concepts and compares them triple by
triple. Every difference must fall into a documented cause; an undocumented one
fails the notebook, because that is either a rendering bug or a new finding.

The check is not symmetric in status. `linkml-convert` is driven by the published
schema and cannot run without editing a published constraint — see
`45_identity_probe.ipynb` and decision D8. It is a second opinion, not an oracle.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
BUILD     = "../build"
REPORTS   = "../reports"

BC_EXPORT = f"{DOWNLOADS}/cdisc_biomedical_concepts_latest.csv"
BC_MODEL  = f"{BUILD}/cosmos_bc_model.patched.yaml"
INSTANCES = f"{ROOT}/cosmos_bc_v1.instances.ttl"

BC_NS = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"
OBO   = "http://purl.obolibrary.org/obo/NCIT_"

# Every predicate on which the two paths may differ, and why.
EXPECTED_DIFFERENCES = {
    "exactMatch":
        "ours only - authored NCIt dual anchor (D2); the schema has no slot for it",
    "identifier":
        "ours only - authored dcterms:identifier carrying the bare C-code (D2)",
    "parentConceptId":
        "ours emits a link, linkml-convert a literal - published range is string (D11)",
    "resultScales":
        "ours emits the T-Box permissible-value IRI, linkml-convert a string (D11)",
    "packageType":
        "ours emits the T-Box permissible-value IRI, linkml-convert a string (D11)",
    "dataType":
        "ours emits the T-Box permissible-value IRI, linkml-convert a string (D11)",
    "coding":
        "ours identifies the coding by system+code (D10); linkml-convert emits a blank node",
}

# Slots that belong to DataElementConcept and not to BiomedicalConcept. On a
# concept used at both layers (D12) the two roles resolve to one node, so ours
# carries them and a BC-only conversion does not.
DEC_ONLY_SLOTS = {"dataType", "exampleSet", "type"}

## Sample

Chosen by shape rather than by hand, so the selection is reproducible and covers
the cases the two paths can disagree on: an external coding, several data element
concepts, a parent link, multiple result scales, and one of the concepts used at
both layers.

In [ ]:
from pathlib import Path

import pandas as pd

export = pd.read_csv(BC_EXPORT, dtype=str, keep_default_na=False)
latest = export.package_date.groupby(export.bc_id).transform("max")
current = export[export.package_date == latest]
bc_level = current.drop_duplicates("bc_id").set_index("bc_id")

identified = bc_level[bc_level.ncit_code != ""]
dec_counts = current[current.dec_id != ""].groupby("bc_id").dec_id.nunique()
coded = set(current[current.code != ""].bc_id)
dual_role = set(identified.ncit_code) & set(current[current.ncit_dec_code != ""].ncit_dec_code)

def first(condition):
    matches = sorted(bc for bc in identified.index if condition(bc))
    if not matches:
        raise RuntimeError("no concept matches the sampling condition")
    return matches[0]

SAMPLE = sorted({
    "C115805",                                                        # 6MWT, LOINC coding, 5 DECs, parent
    first(lambda b: b in coded and dec_counts.get(b, 0) >= 3),         # coding + several DECs
    first(lambda b: dec_counts.get(b, 0) == 0),                        # no DECs
    first(lambda b: not identified.at[b, "parent_bc_id"]),             # no parent
    first(lambda b: identified.at[b, "ncit_code"] in dual_role),       # used at both layers
    first(lambda b: ";" in identified.at[b, "result_scales"]),         # multiple result scales
})

for bc_id in SAMPLE:
    row = identified.loc[bc_id]
    print(f"{bc_id:10s} {row.short_name[:48]:50s} DECs={dec_counts.get(bc_id,0):>2}  "
          f"coding={'y' if bc_id in coded else 'n'}  parent={'y' if row.parent_bc_id else 'n'}")

## Build the instances and the relaxed schema

Same shape as `45_identity_probe.ipynb`: values are lifted to `NCIT:` CURIEs and
the `conceptId` pattern is relaxed, because the published pattern forbids the only
form that converts. Both substitutions are asserted to match exactly once.

In [ ]:
import re
import yaml

CCODE = re.compile(r"^C[0-9]+$")

def lift(value):
    return f"NCIT:{value}" if CCODE.match(value) else value

def split(value):
    return [part.strip() for part in value.split(";") if part.strip()]

def build(bc_id):
    rows = current[current.bc_id == bc_id]
    head = rows.iloc[0]
    instance = {
        "packageDate": head.package_date,
        "packageType": "bc",
        "conceptId": lift(head.bc_id),
        "shortName": head.short_name,
        "definition": head.definition,
        "categories": split(head.bc_categories),
    }
    if head.ncit_code:
        instance["ncitCode"] = head.ncit_code
    if head.parent_bc_id:
        instance["parentConceptId"] = lift(head.parent_bc_id)
    if split(head.synonyms):
        instance["synonyms"] = split(head.synonyms)
    if split(head.result_scales):
        instance["resultScales"] = split(head.result_scales)

    codings, seen = [], set()
    for _, row in rows.iterrows():
        if row.code and (row.code, row.system) not in seen:
            seen.add((row.code, row.system))
            codings.append({"code": row.code, "system": row.system, "systemName": row.system_name})
    if codings:
        instance["coding"] = codings

    decs = []
    for _, row in rows.drop_duplicates("dec_id").iterrows():
        if not row.dec_id or not row.ncit_dec_code:
            continue
        dec = {"conceptId": lift(row.dec_id), "shortName": row.dec_label, "dataType": row.data_type}
        if row.ncit_dec_code:
            dec["ncitCode"] = row.ncit_dec_code
        decs.append(dec)
    if decs:
        instance["dataElementConcepts"] = decs
    return instance


published = Path(BC_MODEL).read_text(encoding="utf-8")
PUBLISHED_SLOT = """  conceptId:
    description: An identifier that uniquely represents an entity
    identifier: true
    range: string
    pattern: "^(C[0-9]+|NEW_[A-Z_]*[0-9]*)$"
    required: true"""
RELAXED_SLOT = """  conceptId:
    description: An identifier that uniquely represents an entity
    identifier: true
    range: uriorcurie
    pattern: "^(NCIT:C[0-9]+|NEW_[A-Z_]*[0-9]*)$"
    required: true"""
if published.count(PUBLISHED_SLOT) != 1:
    raise RuntimeError("the published conceptId slot no longer has the expected shape")

schema_path = Path(BUILD, "compare_schema_relaxed.yaml")
schema_path.write_text(published.replace(PUBLISHED_SLOT, RELAXED_SLOT), encoding="utf-8")

instance_paths = {}
for bc_id in SAMPLE:
    path = Path(BUILD, f"compare_{bc_id}.yaml")
    path.write_text(yaml.safe_dump(build(bc_id), sort_keys=False, allow_unicode=True), encoding="utf-8")
    instance_paths[bc_id] = path
print(f"wrote {len(instance_paths)} instances and the relaxed schema to {BUILD}/")

## Convert, and pull the same concepts out of the deliverable

In [ ]:
import subprocess
import sys

from rdflib import Graph, URIRef

BIN = Path(sys.executable).parent
ours = Graph().parse(INSTANCES, format="turtle")

converted = {}
for bc_id, path in instance_paths.items():
    result = subprocess.run(
        [str(BIN / "linkml-convert"), "--schema", str(schema_path),
         "--target-class", "BiomedicalConcept", "-t", "ttl", str(path)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"{bc_id}: linkml-convert failed\n{result.stderr[-500:]}")
    converted[bc_id] = Graph().parse(data=result.stdout, format="turtle")
    print(f"{bc_id:10s} linkml-convert -> {len(converted[bc_id]):>3} triples")

## Compare the concept node

Predicate-and-object sets on the concept subject itself. Agreement means the two
paths produced the same statement from the same row; a difference must name its
cause.

In [ ]:
import csv
from collections import Counter

def local(term):
    text = str(term)
    return text.rsplit("#", 1)[-1].rsplit("/", 1)[-1]

agree = Counter()
differ = Counter()
rows = []
unexplained = []

for bc_id in SAMPLE:
    subject = URIRef(OBO + bc_level.at[bc_id, "ncit_code"])
    theirs_po = set(converted[bc_id].predicate_objects(subject))
    ours_po = set(ours.predicate_objects(subject))

    for predicate in {p for p, _ in theirs_po} | {p for p, _ in ours_po}:
        name = local(predicate)
        t = {o for p, o in theirs_po if p == predicate}
        o = {x for p, x in ours_po if p == predicate}

        if t == o:
            agree[name] += 1
            status, cause = "agree", ""
        else:
            differ[name] += 1
            status = "differ"
            if bc_level.at[bc_id, "ncit_code"] in dual_role and name in DEC_ONLY_SLOTS:
                cause = ("concept used at both layers (D12): the two roles resolve to one node, "
                         "so ours carries the DataElementConcept type and slots")
            else:
                cause = EXPECTED_DIFFERENCES.get(name, "")
            if not cause:
                unexplained.append((bc_id, name, sorted(map(str, o))[:2], sorted(map(str, t))[:2]))

        rows.append({
            "concept": bc_id,
            "predicate": name,
            "status": status,
            "ours": " | ".join(sorted(str(x) for x in o))[:180],
            "linkml_convert": " | ".join(sorted(str(x) for x in t))[:180],
            "cause": cause,
        })

print(f"{'predicate':22s} {'agree':>6} {'differ':>7}  cause")
causes_by_predicate = {}
for row in rows:
    if row["status"] == "differ":
        causes_by_predicate.setdefault(row["predicate"], set()).add(row["cause"])

for name in sorted(set(agree) | set(differ)):
    cause = " + ".join(sorted(causes_by_predicate.get(name, {""})))
    print(f"{name:22s} {agree[name]:>6} {differ[name]:>7}  {cause[:64]}")
print()
print(f"unexplained differences: {len(unexplained)}")
for u in unexplained:
    print("   ", u)

## Report

In [ ]:
out = Path(REPORTS, "render_path_comparison.csv")
with open(out, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["concept", "predicate", "status", "ours", "linkml_convert", "cause"])
    writer.writeheader()
    writer.writerows(sorted(rows, key=lambda r: (r["status"], r["predicate"], r["concept"])))
print(f"wrote {out} ({len(rows)} comparisons over {len(SAMPLE)} concepts)")

if unexplained:
    raise RuntimeError(
        f"{len(unexplained)} difference(s) between the two render paths are not accounted for. "
        "Each is either a bug in the direct renderer or a new finding about the published schema."
    )

print()
print(f"{sum(agree.values())} predicate comparisons agree; {sum(differ.values())} differ, all documented.")
print()
print("Worth noting which side linkml-convert takes: on enum values and on")
print("parentConceptId it agrees with gen-shacl and not with gen-owl. Two of the")
print("three LinkML generators read the schema as describing a document.")